# 03 · Update, measure, gate, promote

*Part 2, sections 3-4.* The first full pass of the loop, by hand: a person writes v2, both versions
run on the same dataset with the same scorers, and v2 ships only if it measurably helps.

The rules every later level reuses (`selfimprove/gate.py`):

1. **must-pass** scorers are at 100% for the candidate, on both splits
2. the candidate is **not worse** than production on the train split (the CI suite)
3. it **beats production on the holdout** by at least `SI_PROMOTION_MARGIN`
4. it is better somewhere (a no-op never ships), and not bloated

In [ ]:
import _nb as nb

experiment_id = nb.setup()

In [ ]:
import mlflow
from selfimprove import agents, config, gate, instructions, signals, store

print(open("examples/fixes/v2_manual.md").read())

## 1. Register v2 without promoting it

The commit message names the evidence. `production` stays on v1 until v2 has been measured.

In [ ]:
v2 = mlflow.genai.register_prompt(
    name=config.settings.prompt_name,
    template=open("examples/fixes/v2_manual.md").read(),
    commit_message="v2: lead with the answer, always look up the policy, never guess "
                   "(evidence: the stale-refund and disclaimer failures in notebook 01)",
    tags={"author": "you", "status": "candidate"},
)
print(instructions.diff("production", v2.version))

**Under the hood:** `register_prompt()` with an existing name adds version 2 under the same
prompt, with its own tags (`author`, `status=candidate`). No alias moves, so `@production` still
resolves to v1 and live traffic is unaffected. This is what the harness's
`instructions.register_candidate()` does. `instructions.diff()` loads both versions with
`mlflow.genai.load_prompt()` and diffs the text in Python.

In [ ]:
for v in (1, v2.version):
    p = mlflow.genai.load_prompt(f"prompts:/{config.settings.prompt_name}/{v}")
    print(f"v{p.version}  aliases={p.aliases}  tags={p.tags}")

## 2. Measure both on the same data

Each `evaluate()` call is an MLflow evaluation run, so the two sit side by side in the UI with
per-row outcomes and rationales. `predict_fn` pins a version: the agent answers every row fresh.

In [ ]:
train, holdout = store.rows("train"), store.rows("holdout")
scorers = signals.full()
print(len(train), "train rows,", len(holdout), "holdout rows;", [s.name for s in scorers])

v1_train = mlflow.genai.evaluate(data=train, predict_fn=agents.predict_fn(1), scorers=scorers)
v2_train = mlflow.genai.evaluate(data=train, predict_fn=agents.predict_fn(v2.version), scorers=scorers)

In [ ]:
import pandas as pd

def pass_rates(result):
    df = result.result_df
    return {s.name: df[f"{s.name}/value"].map(gate.passed).mean() for s in scorers if f"{s.name}/value" in df}

pd.DataFrame({"v1": pass_rates(v1_train), "v2": pass_rates(v2_train)}).style.format("{:.0%}")

**Under the hood:** with a `predict_fn`, `mlflow.genai.evaluate()` calls the agent once per
row. `agents.predict_fn(v)` wraps `agents.run(question, ref=v, source="eval")`, so every answer is a
new trace tagged `si_source=eval` and the pinned `si_prompt_version`. MLflow links each of those traces
to the evaluation run (trace metadata `mlflow.sourceRun`), logs the scorers' verdicts on them, and
writes the per-scorer averages as run metrics.

In [ ]:
evals = mlflow.search_traces(run_id=v2_train.run_id, return_type="list")
print(len(evals), "traces in the v2 run; si_source:", {t.info.tags.get("si_source") for t in evals},
      "| si_prompt_version:", {t.info.tags.get("si_prompt_version") for t in evals})
print("linked to the run:", all(t.info.trace_metadata.get("mlflow.sourceRun") == v2_train.run_id for t in evals))
print({k: round(float(v), 2) for k, v in v2_train.metrics.items()})

Three things to check before trusting the numbers: the cases that failed should now pass (the
change targeted them), the golden cases should still pass, and the **raw responses** should
actually be good, not just pass:

In [ ]:
for _, row in v2_train.result_df.head(4).iterrows():
    print("Q:", row["request"]["question"])
    print("A:", str(row["response"])[:240], "\n")

## 3. The gate

`gate.compare()` does the above for both splits and applies the rules. It is the same function
that gates the analyst's, the optimizer's and a coding agent's candidates later.

In [ ]:
decision = gate.compare(v2.version, "production")
print(decision.table())
print()
print(decision.summary())

**Under the hood:** `gate.compare()` writes one decision record in MLflow:

* a parent run `gate: v2 vs v1` with tags `si.job=gate`, `si.candidate_version`, `si.baseline_version`
* four nested runs, `train/baseline`, `train/candidate`, `holdout/baseline`, `holdout/candidate`. Each
  is one `mlflow.genai.evaluate(data=rows, predict_fn=agents.predict_fn(version), scorers=...)`, tagged
  `si.label` and `si.prompt_version`
* on the parent: metrics `<split>_<role>_overall` and `prompt_growth`, tags `si.gate_passed` and
  `si.gate_reasons`, and the artifacts `gate_table.txt` and `candidate.diff` (`mlflow.log_text`)
* on the candidate prompt version: tags `gate=passed|failed` and `gate_run=<run id>`

In [ ]:
gate_run = mlflow.get_run(decision.run_id)
print(gate_run.info.run_name, {k: v for k, v in gate_run.data.tags.items() if k.startswith("si.")})
print({k: round(v, 2) for k, v in gate_run.data.metrics.items()})
for r in mlflow.search_runs(filter_string=f"tags.mlflow.parentRunId = '{decision.run_id}'", output_format="list"):
    print(f"  nested {r.info.run_name:<18} si.label={r.data.tags.get('si.label')}  si.prompt_version={r.data.tags.get('si.prompt_version')}")
print("artifacts:", [f.path for f in mlflow.MlflowClient().list_artifacts(decision.run_id)])
cand = mlflow.genai.load_prompt(f"prompts:/{config.settings.prompt_name}/{decision.candidate}")
print("prompt version tags:", {k: v for k, v in cand.tags.items() if k.startswith("gate")})

## 4. The dataset as a regression suite

The same rows run as a pytest suite with `@mlflow.test`, one MLflow run per session, and CI runs
it on every pull request (`.github/workflows/selfimprove-regression.yml`):

In [ ]:
print(open("tests/regression/test_regressions.py").read().split('"""', 2)[2])

```bash
SI_REGRESSION=1 pytest -p mlflow.pytest.plugin tests/regression -n 4
```

## 5. Promote: a pointer move

The agent picks up v2 on its next request, with no code change. The version it replaced becomes
`previous`, so rollback is one alias move too.

In [ ]:
if decision.passed:
    replaced = instructions.promote(v2.version, reason=f"gate run {decision.run_id}", by="you")
    print(f"production: v{replaced} -> v{instructions.load('production').version}")
else:
    print("not promoting:", decision.reasons)

**Under the hood:** `instructions.promote()` makes two `mlflow.genai.set_prompt_alias()` calls:
`previous` to the old version, then `production` to the new one. It records why with
`mlflow.genai.set_prompt_version_tag()`: the new version gets `status=production`, `promoted_by`,
`promoted_because`, `promoted_at` and `replaced_version`; the old one gets `status=retired`. The agent
resolves `@production` on its next call, which is why no code changes. `instructions.rollback()` is
the reverse alias move.

In [ ]:
for v in (1, v2.version):
    p = mlflow.genai.load_prompt(f"prompts:/{config.settings.prompt_name}/{v}")
    print(f"v{p.version}  aliases={p.aliases}  " + str({k: p.tags.get(k) for k in ("status", "promoted_by", "replaced_version")}))

In [ ]:
print(agents.run("How long can I wait to request a refund?", source="production"))

One pass produced two prompt versions, a dataset, two evaluation runs that show the before and
after, and a regression suite. Every step was manual.

**Next:** [04 · Triage and approve](04_triage_and_approve.ipynb), for when there is too much signal
to read by hand.